# Notebook 10: torch.func Transforms

## Why This Matters

`torch.func` (formerly `functorch`) brings JAX-style functional transforms to PyTorch. The critical use cases that are otherwise painful or impossible with standard autograd: computing per-sample gradients for differentially private training (DP-SGD) requires the gradient for each sample individually, not just the batch average -- `vmap(grad(...))` does this in one line. MAML (Model-Agnostic Meta-Learning) requires differentiating through the gradient computation step itself -- `grad(grad(...))` with `create_graph=True` via `torch.func.grad`. These are not academic curiosities; they are requirements for modern privacy-preserving ML and few-shot learning systems.

## Background

Functional transforms operate on **pure functions** -- functions that take tensors as inputs and return tensors as outputs, with no side effects (no `.grad` accumulation, no module state). This is why `torch.func` APIs work on functions rather than `nn.Module` objects directly.

The four main transforms:
- `torch.func.grad(f)` -- functional gradient, returns a function that computes $\nabla f$
- `torch.func.vmap(f)` -- vectorises $f$ over a batch dimension (no Python loop)
- `torch.func.jacrev(f)` -- Jacobian via reverse-mode AD
- `torch.func.jacfwd(f)` -- Jacobian via forward-mode AD (better for wide functions)

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import grad, vmap, jacrev, jacfwd, functional_call
import numpy as np
import matplotlib.pyplot as plt

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

# Verify torch.func availability
import torch.func
print(f"torch.func available: {hasattr(torch, 'func')}")
print("Ready.")

## 1. torch.func.grad -- Functional Gradient

`torch.func.grad(f)` returns a function that computes the gradient of scalar function `f` with respect to its first argument. Unlike `loss.backward()`, it does NOT populate `.grad` attributes -- it returns gradient tensors directly. This is the functional, side-effect-free interface.

In [ ]:
# Basic torch.func.grad
def f(x):
    return (x ** 2).sum()  # scalar output required

# grad(f) returns a function that computes df/dx
grad_f = grad(f)

x = torch.tensor([1.0, 2.0, 3.0])
g = grad_f(x)
print(f"f(x) = sum(x^2)")
print(f"grad(f)(x) = {g}  (expected: 2x = [2, 4, 6])")

# x.grad is NOT populated
print(f"x.grad = {x.grad}  (None -- functional, no side effects)")

# argnums: differentiate w.r.t. a different argument
def loss_fn(W, b, x, y_true):
    logits = x @ W + b
    return F.cross_entropy(logits, y_true)

W = torch.randn(8, 3)
b = torch.randn(3)
x = torch.randn(4, 8)
y = torch.randint(0, 3, (4,))

# Gradient w.r.t. W (arg 0)
dW = grad(loss_fn, argnums=0)(W, b, x, y)
print(f"\ndW shape: {dW.shape}  (same as W: {W.shape})")

# Gradient w.r.t. both W and b
dW, db = grad(loss_fn, argnums=(0, 1))(W, b, x, y)
print(f"dW shape: {dW.shape}, db shape: {db.shape}")

## 2. torch.func.vmap -- Vectorise Over a Batch Dimension

`vmap(f)` transforms a function `f` that operates on a single example into one that operates on a batch, without writing explicit loops. Under the hood it uses a batched autograd pass rather than a Python for loop, making it significantly faster for large batches.

The mental model: `vmap(f)(xs)` is equivalent to `torch.stack([f(xs[i]) for i in range(len(xs))])` but implemented as a single vectorised op.

In [ ]:
# vmap: vectorise a single-sample function over a batch
def dot_product(a, b):
    return (a * b).sum()  # scalar, for one pair of vectors

a_batch = torch.randn(32, 16)  # 32 samples, each is a 16-d vector
b_batch = torch.randn(32, 16)

# Manual loop (slow for large batches)
loop_result = torch.stack([dot_product(a_batch[i], b_batch[i]) for i in range(32)])

# vmap (vectorised, fast)
batched_dot = vmap(dot_product)
vmap_result = batched_dot(a_batch, b_batch)

print(f"Loop result shape: {loop_result.shape}")
print(f"vmap result shape: {vmap_result.shape}")
print(f"Results match: {torch.allclose(loop_result, vmap_result)}")

# vmap over a specific axis (in_dims)
# a is (32, 16), b is always the same (16,)
fixed_b = torch.randn(16)
batched_dot_fixed = vmap(dot_product, in_dims=(0, None))  # None: b is NOT batched
result_fixed = batched_dot_fixed(a_batch, fixed_b)
print(f"\nvmap with fixed b: {result_fixed.shape}  (dot of each row with fixed_b)")

## 3. Per-Sample Gradients with vmap(grad(loss_fn))

The canonical DP-SGD use case: for differentially private training, you need to clip each sample's gradient **individually** before aggregating. Standard `loss.backward()` only gives you the batch-averaged gradient. `vmap(grad(per_sample_loss))` gives you a gradient tensor per sample in one efficient pass.

In [ ]:
# Per-sample gradients for DP-SGD
model = nn.Sequential(nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 4))
params = dict(model.named_parameters())

# functional_call: run model as a pure function of params + input
def single_sample_loss(params, x, y):
    logits = functional_call(model, params, (x.unsqueeze(0),)).squeeze(0)
    return F.cross_entropy(logits, y.unsqueeze(0))

# grad of single-sample loss w.r.t. params
per_sample_grad = grad(single_sample_loss)

# Vectorise over the batch dimension
batch_per_sample_grad = vmap(per_sample_grad, in_dims=(None, 0, 0))

X_dp = torch.randn(8, 16)   # 8 samples
y_dp = torch.randint(0, 4, (8,))

# Compute per-sample gradients
per_sample_grads = batch_per_sample_grad(params, X_dp, y_dp)
print("Per-sample gradient shapes:")
for name, g in per_sample_grads.items():
    print(f"  {name}: {g.shape}  (batch, *param_shape)")

# Clip each sample's gradient independently (L2 norm clipping)
MAX_NORM = 1.0
clipped_grads = {}
for name, g in per_sample_grads.items():
    flat_g = g.flatten(1)  # (batch, param_elements)
    norms = flat_g.norm(2, dim=1, keepdim=True)  # (batch, 1)
    scale = (MAX_NORM / (norms + 1e-8)).clamp(max=1.0)
    clipped_grads[name] = (flat_g * scale).view_as(g).mean(0)

print(f"\nAggregated clipped gradient shapes:")
for name, g in clipped_grads.items():
    print(f"  {name}: {g.shape}  (same as param)")

## 4. Jacobians with jacrev and jacfwd

The Jacobian $J_{ij} = \partial f_i / \partial x_j$ gives the full sensitivity matrix: how much each output dimension changes with respect to each input dimension. Applications:
- **Sensitivity analysis**: which input features most affect each output?
- **Neural Tangent Kernel (NTK)**: Jacobian of model w.r.t. parameters
- **Pruning analysis**: which parameters have the least effect on outputs?

`jacrev` (reverse-mode) is efficient when outputs << inputs. `jacfwd` (forward-mode) is efficient when inputs << outputs.

In [ ]:
def softmax_fn(x):
    return F.softmax(x, dim=-1)

x_jac = torch.randn(5)  # 5-d input -> 5-d softmax output

# Jacobian of softmax: (output_dim, input_dim) = (5, 5)
J_rev = jacrev(softmax_fn)(x_jac)
J_fwd = jacfwd(softmax_fn)(x_jac)
print(f"jacrev Jacobian shape: {J_rev.shape}")
print(f"jacfwd Jacobian shape: {J_fwd.shape}")
print(f"J_rev == J_fwd: {torch.allclose(J_rev, J_fwd, atol=1e-5)}")

print("\nSoftmax Jacobian (row i, col j = d softmax_i / d x_j):")
print(J_rev.detach().numpy().round(4))

# Verify: rows of Jacobian sum to 0 (softmax outputs sum to 1)
print(f"\nRow sums (should be 0): {J_rev.sum(1).tolist()}")

# Efficiency rule:
# jacrev: O(out_dim) backward passes
# jacfwd: O(in_dim) forward passes
# Rule of thumb: jacrev if out < in, jacfwd if in < out
print(f"\nFor in=5, out=5: either works equally")
print(f"For in=100, out=5: use jacrev (5 passes vs 100 passes)")
print(f"For in=5, out=100: use jacfwd (5 passes vs 100 passes)")

## 5. Higher-Order Gradients -- MAML Pattern

Model-Agnostic Meta-Learning (MAML) requires differentiating through a gradient update step. The inner loop computes updated parameters by taking a gradient step; the outer loop minimises the loss AFTER those updates. This requires computing gradients of gradients.

In [ ]:
# MAML inner-loop: take one gradient step, return updated params
def inner_update(params, x_support, y_support, inner_lr=0.1):
    def loss_for_params(p):
        logits = functional_call(model, p, (x_support,))
        return F.cross_entropy(logits, y_support)

    # Gradient of loss w.r.t. params
    grads = grad(loss_for_params)(params)
    # SGD step: theta' = theta - alpha * grad
    updated_params = {k: params[k] - inner_lr * grads[k] for k in params}
    return updated_params

# Outer objective: loss on query set AFTER the inner update
def outer_loss(params, x_support, y_support, x_query, y_query):
    updated = inner_update(params, x_support, y_support)
    logits_query = functional_call(model, updated, (x_query,))
    return F.cross_entropy(logits_query, y_query)

# Setup
model_maml = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 4))
params_maml = dict(model_maml.named_parameters())

x_sup = torch.randn(8, 8); y_sup = torch.randint(0, 4, (8,))
x_qry = torch.randn(8, 8); y_qry = torch.randint(0, 4, (8,))

# Compute meta-gradient: d(outer_loss) / d(original_params)
meta_grads = grad(outer_loss)(params_maml, x_sup, y_sup, x_qry, y_qry)
print("MAML meta-gradients (gradients through the gradient step):")
for name, g in meta_grads.items():
    print(f"  {name}: {g.shape}  norm={g.norm():.4f}")
print("\nThese gradients flow back through the inner update -- 2nd order!")

## Summary

### Key Concepts

| Concept | Key Point |
|---------|----------|
| torch.func.grad | Functional gradient; no .grad side effects; works on pure functions |
| vmap | Vectorises a single-sample function over a batch; faster than Python loops |
| vmap(grad(...)) | Per-sample gradients for DP-SGD; gradient per sample, not batch average |
| jacrev | Jacobian via reverse-mode; efficient when out_dim << in_dim |
| jacfwd | Jacobian via forward-mode; efficient when in_dim << out_dim |
| functional_call | Run nn.Module as a pure function of params + input; needed for vmap/grad |
| MAML pattern | grad(grad(loss)) -- meta-gradient through a gradient step |

### Common Pitfalls
- Using `torch.func.grad` on a function that has side effects -- results are undefined
- Forgetting `functional_call` when using vmap with nn.Module -- modules are not batchable directly
- Using `jacrev` when `jacfwd` would be 10x faster (in_dim < out_dim)
- Missing the `argnums` parameter -- differentiates w.r.t. wrong argument
- Not normalising per-sample gradients before summing -- effective LR becomes batch-size-dependent

### Next: Notebook 11 -- Distributed Data Parallel